In [98]:
import numpy as np

# Define physical system size 
L = 3

# Exchange (swap) operator P12 for two qubits
P12 = np.array([[1, 0, 0, 0],
                [0, 0, 1, 0],
                [0, 1, 0, 0],
                [0, 0, 0, 1]])

# Identity operator I for the two-particle system
I = np.eye(4)

# Symmetrization operator S (for bosons)
S = 0.5 * (I + P12)

# Antisymmetrization operator A (for fermions)
A = (I - P12) * 0.5

single_spin_basis_states = [ np.eye(2**2)[i] for i in range(2**2)]

def direct_sum(A, B):
    # Get the shapes of the input matrices
    rows_A, cols_A = A.shape
    rows_B, cols_B = B.shape
    
    # Create a zero matrix with the combined shape
    result = np.zeros((rows_A + rows_B, cols_A + cols_B))
    
    # Place matrix A in the top-left block
    result[:rows_A, :cols_A] = A
    
    # Place matrix B in the bottom-right block
    result[rows_A:, cols_A:] = B
    
    return result

def recursive_tensordot(tensors, axes=0):
    # this function used to compute the collective tensor product of a list of rank 1 tensors
    if len(tensors) == 1:
        # Base case: only one tensor, return it
        return tensors[0]
    elif len(tensors) == 2:
        # Base case: two tensors, directly apply np.tensordot
        return np.tensordot(tensors[0], tensors[1], axes=axes).reshape(-1)
    else:
        # Recursive case: combine the first two tensors and recurse
        result = np.tensordot(tensors[0], tensors[1], axes=axes)
        return recursive_tensordot([result] + tensors[2:], axes=axes).reshape(-1)
    
def recursive_kron(tensors):
    # this function used to compute the collective kronecker product of a list of rank 2 tensors
    if len(tensors) == 1:
        # Base case: only one tensor, return it
        return tensors[0]
    elif len(tensors) == 2:
        # Base case: two tensors, directly apply np.tensordot
        return np.kron(tensors[0], tensors[1])
    else:
        # Recursive case: combine the first two tensors and recurse
        result = np.kron(tensors[0], tensors[1])
        return recursive_kron([result] + tensors[2:])
    
def transformation_matrix(L):

    n = 2 ** (2 * L)
    old_basis_indices = [n for n in range(n)]
    # Initialize dictionary to store lists of integers modulo L
    result = {i: [] for i in range(L)}
    
    # Factor the list of integers
    for num in old_basis_indices:
        residue = num % L
        result[residue].append(num)

    new_basis_indices = sum([result[key] for key in result.keys()], [])
    l = [old_basis_indices, new_basis_indices]
    coordinates = [[row[i] for row in l] for i in range(len(l[0]))]

    # initialize U
    U = np.zeros((n,n))
    for coordinate in coordinates:
        U[coordinate[1], coordinate[0]] += 1
    
    return U

component_states = [np.tensordot(A, single_spin_basis_states[1], axes = 1) for i in range(L)]
# component_state is a list of all the single-spin antisymmetric state waiting to be tensored together

U = transformation_matrix(L)
pre_vbs = np.tensordot(U, recursive_tensordot(component_states), axes = 1) # after transforming to the symmetric basis
component_symmetric_operators = [S for i in range(L)]
tensor_product_symmetric_operators = recursive_kron(component_symmetric_operators)
vbs = np.tensordot(tensor_product_symmetric_operators, pre_vbs, axes = 1)
print(vbs)
print([np.binary_repr(n, width = 2 * L) for n in np.nonzero(vbs)[0]])


[ 0.      -0.0625  -0.0625   0.       0.      -0.03125 -0.03125  0.
  0.      -0.03125 -0.03125  0.       0.       0.0625   0.0625   0.
  0.       0.       0.       0.       0.       0.       0.       0.
  0.       0.       0.       0.       0.      -0.03125 -0.03125  0.
  0.       0.       0.       0.       0.       0.       0.       0.
  0.       0.       0.       0.       0.      -0.03125 -0.03125  0.
  0.       0.0625   0.0625   0.       0.       0.03125  0.03125  0.
  0.       0.03125  0.03125  0.       0.      -0.0625  -0.0625   0.125  ]
['000001', '000010', '000101', '000110', '001001', '001010', '001101', '001110', '011101', '011110', '101101', '101110', '110001', '110010', '110101', '110110', '111001', '111010', '111101', '111110', '111111']
